In [24]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [25]:
sentences = [
    "a young hero discovers hidden powers and saves the world",
    "two strangers fall in love during a dangerous adventure",
    "a detective solves a mysterious murder in a dark city",
    "an astronaut gets stranded alone on a distant planet",
    "a family escapes a haunted house full of dark secrets",
    "a soldier returns home and struggles to find peace",
    "a scientist creates an artificial intelligence that turns evil",
    "a thief plans the biggest heist in history",
    "a young girl discovers a magical world beneath the ocean",
    "a team of heroes fights to stop a global catastrophe",
    "a spy uncovers a dangerous conspiracy inside the government",
    "a robot learns what it means to be truly human",
    "a child befriends an alien stranded far from home",
    "a musician chases his dream against all the odds",
    "a king loses his throne and fights to reclaim it",
    "a journalist exposes corruption at the highest levels of power",
    "a survivor escapes a deadly island filled with monsters",
    "twin brothers separated at birth finally find each other",
    "a time traveler tries to fix mistakes from the past",
    "a retired agent comes back for one final dangerous mission",
]

In [26]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
vocab_size = len(tokenizer.word_index) + 1

input_sequences = []
for sentence in sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_len = max(len(s) for s in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = tf.keras.utils.to_categorical(input_sequences[:, -1], num_classes=vocab_size)

print(f"Vocab size : {vocab_size}")
print(f"Max seq len: {max_len}")
print(f"X shape    : {X.shape}")
print(f"y shape    : {y.shape}")

Vocab size : 132
Max seq len: 10
X shape    : (169, 9)
y shape    : (169, 132)


In [27]:
inputs = Input(shape=(max_len - 1,))
x = Embedding(vocab_size, 64)(inputs)
x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Dropout(0.3)(x)
x = Bidirectional(LSTM(64))(x)
x = Dropout(0.3)(x)
outputs = Dense(vocab_size, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 9, 64)          │         8,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 9, 256)         │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 9, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 132)            │        17,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 387,460 (1.48 MB)

 Trainable params: 387,460 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
history = model.fit(X, y, epochs=100, verbose=1)

Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 44ms/step - accuracy: 0.0355 - loss: 4.8839
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.0592 - loss: 4.8737
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.0533 - loss: 4.8567
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - accuracy: 0.0533 - loss: 4.8363
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - accuracy: 0.0533 - loss: 4.7854
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 93ms/step - accuracy: 0.0533 - loss: 4.7343
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.0592 - loss: 4.6710
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step - accuracy: 0.0828 - loss: 4.5965
Epoch 9/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step - accuracy: 0.0769 - loss: 4.4494
Epoch 10/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.0888 - loss: 4.3580
Epoch 11/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.0947 - loss: 4.3557
Epoch 12/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.1006 - lo

In [29]:
def complete_text(seed_text, next_words=6):
    print(f"\nInput : {seed_text}")
    result = seed_text
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([result])[0]
        token_list = pad_sequences([token_list], maxlen=max_len - 1, padding='pre')
        probs      = model.predict(token_list, verbose=0)[0]
        next_idx   = np.argmax(probs)
        next_word  = tokenizer.index_word.get(next_idx, '')
        if not next_word:
            break
        result += ' ' + next_word
    print(f"Output: {result}")

complete_text("a young hero")
complete_text("a detective solves")
complete_text("an astronaut gets")
complete_text("a spy uncovers")
complete_text("a time traveler")
complete_text("a retired agent")



Input : a young hero
Output: a young hero discovers hidden powers and saves the

Input : a detective solves
Output: a detective solves a mysterious murder in a dark

Input : an astronaut gets
Output: an astronaut gets stranded alone on a distant planet

Input : a spy uncovers
Output: a spy uncovers a dangerous conspiracy inside the government

Input : a time traveler
Output: a time traveler tries to fix mistakes from the

Input : a retired agent
Output: a retired agent comes back for one final dangerous
